# Diabetes Hospital Readmission Analysis

## Overview
Analysis of clinical data from 1999-2008 across 130 US hospitals to identify patient profiles at high risk of readission among diabetic patients. 

## Dataset
- Source: UCI 
- Records: 100,000+ Rows
- Target Variable: Readmission status - No readmission, Less than 30 days, More than 30 days

## Tools
- Platform: Databricks
- Language: SQL (Exploratory Data Analysis) + Python (Modeling)



In [0]:
USE diabetesdb

In [0]:
SELECT *
from diabetes_data
LIMIT 10;

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
35754,82637451,Caucasian,Male,[50-60),?,2,1,2,3,?,?,31,6,16,0,0,0,414,411,250,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
55842,84259809,Caucasian,Male,[60-70),?,3,1,2,4,?,?,70,1,21,0,0,0,414,411,V45,7,None,None,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
63768,114882984,Caucasian,Male,[70-80),?,1,1,7,5,?,?,73,0,12,0,0,0,428,492,250,8,None,None,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
12522,48330783,Caucasian,Female,[80-90),?,2,1,4,13,?,?,68,2,28,0,0,0,398,427,38,8,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
15738,63555939,Caucasian,Female,[90-100),?,3,3,4,12,?,InternalMedicine,33,3,18,0,0,0,434,198,486,8,None,None,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 1. Data Overview
Patient readmission distribution

In [0]:
SELECT
COUNT (*) as total_encounters,
SUM(CASE WHEN readmitted = 'NO' THEN 1 ELSE 0 END) as no_readmission,
SUM(CASE WHEN readmitted = '>30' THEN 1 ELSE 0 END) as readmitted_after_30,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_before_30
FROM diabetes_data;

total_encounters,no_readmission,readmitted_after_30,readmitted_before_30
101766,54864,35545,11357


From 101766 patient encounters:
- 54,864 were not readmitted
- 35,545 were readmitted after 30 days
- 11,357 were readmitted before 30 days

Patients readmitted before the 30-day mark suggest a high-risk patient profile, possibly treatment failure.

## 2. Readmission by age group before 30 days

In [0]:
--Do older patients face more readmissions?
SELECT
age,
COUNT(*) as total_encounters,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_before_30,
ROUND(SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) * 100 / count(*), 1) as pct_readmitted_before_30
FROM diabetes_data
GROUP BY age 
ORDER BY age;

age,total_encounters,readmitted_before_30,pct_readmitted_before_30
[0-10),161,3,1.9
[10-20),691,40,5.8
[20-30),1657,236,14.2
[30-40),3775,424,11.2
[40-50),9685,1027,10.6
[50-60),17256,1668,9.7
[60-70),22483,2502,11.1
[70-80),26068,3069,11.8
[80-90),17197,2078,12.1
[90-100),2793,310,11.1


### Observations
The older patients being readmitted within 30 days makes intuitive sense and what one would think, however, the data shows notably that patients between the ages of [20-30] have the highest rate of readmissions at 14.2%

## 3. Readmission by demographics
How does readmission vary by demographics

In [0]:
SELECT race,
COUNT(*) as total_encounters,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_before_30,
ROUND(SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) * 100 / count(*), 1) as pct_readmitted_before_30
FROM diabetes_data
GROUP BY race
ORDER BY pct_readmitted_before_30 DESC;

race,total_encounters,readmitted_before_30,pct_readmitted_before_30
Caucasian,76099,8592,11.3
AfricanAmerican,19210,2155,11.2
Hispanic,2037,212,10.4
Asian,641,65,10.1
Other,1506,145,9.6
?,2273,188,8.3


### Observations
Readmission rates within 30 days by racial groups show minimal variation, revealing that race alone is not a strong indicator of early readmission.

## 4. Readmission by Time in Hospital

In [0]:
SELECT time_in_hospital,
COUNT(*) as total_encounters,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_before_30,
ROUND(SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) * 100 / count(*), 1) as pct_readmitted_before_30
FROM diabetes_data
GROUP BY time_in_hospital
ORDER BY time_in_hospital;

time_in_hospital,total_encounters,readmitted_before_30,pct_readmitted_before_30
1,14208,1162,8.2
2,17224,1712,9.9
3,17756,1894,10.7
4,13924,1644,11.8
5,9966,1199,12.0
6,7539,949,12.6
7,5859,752,12.8
8,4391,625,14.2
9,3002,412,13.7
10,2342,336,14.3


### Observations
Time in hospital, aside from patients staying a day or two, tends to show the same early readmission rate, while the peak is around days 8-10. The data suggests an uptrend, but not a linear relationship, as the longer stays have a sort of diminishing returns.
Time in hospital alone is not an accurate predictor for early admissions.

## 5. Readmission by Number of Medications
Are patients who take more medication at a higher risk of early readmission

In [0]:
SELECT num_medications,
COUNT(*) as total_encounters,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_before_30,
ROUND(SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) * 100 / count(*), 1) as pct_readmitted_before_30
FROM diabetes_data
GROUP BY num_medications
ORDER BY num_medications;

num_medications,total_encounters,readmitted_before_30,pct_readmitted_before_30
1,262,11,4.2
2,470,39,8.3
3,900,65,7.2
4,1417,114,8.0
5,2017,150,7.4
6,2699,229,8.5
7,3484,305,8.8
8,4353,416,9.6
9,4913,482,9.8
10,5346,533,10.0


### Observations
Number of medications and early admission rates do appear to have an upward trend, with patients on less than 10 medications showing less than a 10% rate; meanwhile, patients on 20+ medications show a consistent 12%+ rate.

- num_medications reflects distinct medications per hospital visit
- A high number of medications are low in sample size, so considered not statistically reliable to be taken into account.

The number of medications appears to be a more reliable indicator of early readmission compared to either race or time spent in the hospital.

## 6. Readmission by Insulin
Do patients on insulin reveal higher early readmission rates

In [0]:
SELECT insulin,
count(*) as total_encounters,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_before_30,
ROUND(SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) * 100 / count(*), 1) AS pct_readmitted_before_30
FROM diabetes_data
GROUP BY insulin
ORDER BY pct_readmitted_before_30  ;


insulin,total_encounters,readmitted_before_30,pct_readmitted_before_30
No,47383,4756,10.0
Steady,30849,3433,11.1
Up,11316,1470,13.0
Down,12218,1698,13.9


### Observations
Patients not on insulin show the lowest rate of early readmission at 10%, while those with insulin vary from 11 to 14%. The direction of insulin change is not a standout prediction alone, as suggested by the difference between an increase and a decrease in dosage being ~1%. 

## 7. Readmission by Diagnoses
Do patients with more diagnoses experience a higher rate of early readimssion

In [0]:
SELECT number_diagnoses,
count(*) AS total_encounters,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_before_30,
ROUND(SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) * 100 / count(*), 1) AS pct_readmitted_before_30
FROM diabetes_data
GROUP BY number_diagnoses
ORDER BY number_diagnoses;



number_diagnoses,total_encounters,readmitted_before_30,pct_readmitted_before_30
1,219,13,5.9
2,1023,62,6.1
3,2835,209,7.4
4,5537,457,8.3
5,11393,1043,9.2
6,10161,1058,10.4
7,10393,1119,10.8
8,10616,1254,11.8
9,49474,6125,12.4
10,17,3,17.6


### Observations
An uptrend seems to exist between the number of diagnoses and the early admission rate, rising from 5.9% at 1 diagnosis to 12.4% at 9 diagnoses. Similarly to num_medications, diagnoses with more than 9 have small sample sizes, making them statistically unreliable. 

**Note**: The value of 9 diagnoses appears to contain nearly half the dataset, suggesting that the system had a cap on the number of diagnoses able to be recorded, and at some point, it did not, which can be explained by the higher values having a very small sample size. 

## 8. High Risk Profiles based on data
Accounting for all findings into a single query using a CTE to identify the patterns that are linked with early readmission.

In [0]:
WITH patient_risk AS (
  SELECT
  age,
  insulin,
  number_diagnoses,
  num_medications,
  time_in_hospital,
  readmitted,
  CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END as early_readmission
  FROM diabetes_data
)
SELECT
age,
insulin,
ROUND(AVG(number_diagnoses), 1) AS avg_diagnoses,
ROUND(AVG(num_medications), 1) AS avg_medications,
ROUND(AVG(time_in_hospital), 1) AS avg_time_in_hospital,
COUNT(*) AS total_encounters,
SUM(early_readmission) as readmitted_before_30,
ROUND(SUM(early_readmission) * 100 / COUNT(*), 1) AS pct_readmitted_before_30
FROM patient_risk
WHERE insulin != 'No'
GROUP BY age, insulin
HAVING COUNT(*) > 100
ORDER BY pct_readmitted_before_30 DESC
LIMIT 15;

age,insulin,avg_diagnoses,avg_medications,avg_time_in_hospital,total_encounters,readmitted_before_30,pct_readmitted_before_30
[20-30),Up,6.0,13.1,3.8,367,77,21.0
[20-30),Down,6.0,12.9,3.6,364,71,19.5
[80-90),Down,8.2,18.6,5.3,1727,287,16.6
[90-100),Down,8.2,17.7,5.5,261,39,14.9
[30-40),Down,6.8,15.7,4.1,601,87,14.5
[70-80),Down,8.0,19.9,5.3,2756,382,13.9
[90-100),Up,8.2,17.1,5.4,279,38,13.6
[60-70),Down,7.9,20.6,5.1,2675,364,13.6
[40-50),Down,7.3,18.0,4.5,1451,196,13.5
[70-80),Up,8.1,20.5,5.6,2569,345,13.4


### Observations

The following analysis highlights two high-risk patient profiles:

**Young Patients [20,30)** on insulin adjustments show the highest rate of early readmission with 19.5 to 21%, although they face lower medical complexity with fewer diagnoses and medications. This could suggest that young patients face poor discharge planning or patient compliance.

**Older Patients [70,90)** also show early readmission rates in between 13-17%, however, unlike young patients, older patients are far more medically complex with higher average diagnoses and medications.



## SQL EDA Summary
Analysis of the diabetes data set containing 101,766 diabetic patients across 130 US hospitals revealed the following patterns visible in possible predictions for early readmission:
- Patients aged 20-30 have the highest percentage (14.2%) of readmissions within 30 days, higher than older groups.
- Race does not seem to be a meaningful predictor of early readmission.
- Time in hospital does show an upward trend; however, it faces diminishing returns after 10 days.
- Patients with 10 or more medications have the highest percentage of readmissions within 30 days.
- Patients who face active insulin changes tend to be at a higher risk of early readmission.
- Patients with more diagnoses (possible system cap of 9) tend to face higher early readmission rates.

Key Finding: Two high-risk profiles were identified, which suggest that medical complexity and age are strong indicators of early readmission.